# Лабораторная работа: Ансамбли моделей машинного обучения
**Выполнил:** Хрипков Тимофей, группа ИУ5-65Б

## Цель лабораторной работы
Изучение ансамблей моделей машинного обучения.

## Задание
1. Выбрать набор данных для решения задачи классификации.
2. Провести удаление или заполнение пропусков и кодирование категориальных признаков.
3. Разделить выборку на обучающую и тестовую.
4. Обучить ансамблевые модели:
   - Две модели группы бэггинга (Bagging и Random Forest)
   - AdaBoost
   - Градиентный бустинг
5. Оценить качество моделей с помощью подходящей метрики и сравнить их.

## Описание выбранного датасета
Для работы выбран набор данных **Digits** (распознавание рукописных цифр).  
**Целевая переменная:** `target` — цифра от 0 до 9 (10 классов).  
**Признаки:** 64 числовых признака (8x8 пикселей, значения от 0 до 16). Данные не содержат пропусков и представлены только числовыми переменными.

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Библиотеки для машинного обучения
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

# Настройка
import warnings
warnings.filterwarnings('ignore')

# Загрузка набора данных

In [ ]:
# Загрузка данных Digits
print("Загрузка данных Digits...")
digits = load_digits()
X = digits.data
y = digits.target

df = pd.DataFrame(X, columns=digits.feature_names)
df['target'] = y

print(f"Размерность данных: {df.shape}")
print("\nПервые 5 строк признаков:")
display(df.head())
print(f"\nЦелевая переменная: цифры от 0 до 9")
print(f"Распределение классов:\n{pd.Series(y).value_counts().sort_index().to_dict()}")

Загрузка данных Digits...
Размерность данных: (1797, 65)

Первые 5 строк признаков:


,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,pixel_6_7,pixel_7_0,pixel_7_1,pixel_7_2,pixel_7_3,pixel_7_4,pixel_7_5,pixel_7_6,pixel_7_7,target
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,6.0,13.0,10.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,11.0,16.0,10.0,0.0,0.0,1
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,11.0,16.0,9.0,0.0,2
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,0.0,0.0,0.0,7.0,13.0,13.0,9.0,0.0,0.0,3
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,16.0,4.0,0.0,0.0,4



Целевая переменная: цифры от 0 до 9
Распределение классов:
{0: 178, 1: 182, 2: 177, 3: 183, 4: 181, 5: 182, 6: 181, 7: 179, 8: 174, 9: 180}


# Предварительная обработка данных

## Особенности датасета
- **Пропуски отсутствуют** — не требуется заполнение
- **Все признаки числовые** — не требуется кодирование категориальных переменных
- Данные готовы к использованию без дополнительной обработки

In [ ]:
print(f"Размерность признаков: {X.shape}")
print(f"Размерность целевой переменной: {y.shape}")
print(f"Пропуски в данных: {pd.DataFrame(X).isnull().sum().sum()}")
print(f"Количество классов: {len(np.unique(y))}")

Размерность признаков: (1797, 64)
Размерность целевой переменной: (1797,)
Пропуски в данных: 0
Количество классов: 10


# Разделение выборки

Разделяем данные на обучающую и тестовую выборки в пропорции 80 на 20.

In [ ]:
# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Размер обучающей выборки: {X_train.shape[0]} объектов")
print(f"Размер тестовой выборки: {X_test.shape[0]} объектов")
print(f"Уникальные классы в обучающей: {np.unique(y_train)}")
print(f"Уникальные классы в тестовой: {np.unique(y_test)}")

Размер обучающей выборки: 1437 объектов
Размер тестовой выборки: 360 объектов
Уникальные классы в обучающей: [0 1 2 3 4 5 6 7 8 9]
Уникальные классы в тестовой: [0 1 2 3 4 5 6 7 8 9]


# Обучение ансамблевых моделей

## Модели группы бэггинга
- **BaggingClassifier** — базовый бэггинг на основе деревьев решений
- **RandomForestClassifier** — случайный лес (улучшенный бэггинг с рандомизацией признаков)

## Модели бустинга
- **AdaBoostClassifier** — адаптивный бустинг
- **GradientBoostingClassifier** — градиентный бустинг (обычно более мощный)

In [ ]:
# Инициализация моделей
bagging_model = BaggingClassifier(random_state=42, n_estimators=100)
random_forest_model = RandomForestClassifier(random_state=42, n_estimators=100)
adaboost_model = AdaBoostClassifier(random_state=42, n_estimators=100)
gradient_boosting_model = GradientBoostingClassifier(random_state=42, n_estimators=100)

# Обучение моделей
print("Обучение BaggingClassifier...")
bagging_model.fit(X_train, y_train)

print("Обучение RandomForestClassifier...")
random_forest_model.fit(X_train, y_train)

print("Обучение AdaBoostClassifier...")
adaboost_model.fit(X_train, y_train)

print("Обучение GradientBoostingClassifier...")
gradient_boosting_model.fit(X_train, y_train)

print("\nВсе модели успешно обучены!")

Обучение BaggingClassifier...
Обучение RandomForestClassifier...
Обучение AdaBoostClassifier...
Обучение GradientBoostingClassifier...

Все модели успешно обучены!


# Оценка качества моделей

Для оценки качества в задаче классификации будем использовать:
- **Accuracy** (доля правильных ответов) — основная метрика
- **F1-score** (среднее гармоническое между точностью и полнотой) — для многоклассовой классификации используем `weighted` average

In [ ]:
# Сбор результатов
models = {
    "Bagging": bagging_model,
    "Random Forest": random_forest_model,
    "AdaBoost": adaboost_model,
    "Gradient Boosting": gradient_boosting_model
}

results = []

print("Результаты оценки моделей на тестовой выборке:")
print("-" * 50)

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1-score": f1
    })
    print(f"{name}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print()

results_df = pd.DataFrame(results)
print("\n=== Сводная таблица результатов ===")
print(results_df.to_string(index=False))

Результаты оценки моделей на тестовой выборке:
--------------------------------------------------
Bagging:
  Accuracy: 0.9250
  F1-score: 0.9241

Random Forest:
  Accuracy: 0.9611
  F1-score: 0.9609

AdaBoost:
  Accuracy: 0.8306
  F1-score: 0.8291

Gradient Boosting:
  Accuracy: 0.9556
  F1-score: 0.9553


=== Сводная таблица результатов ===
            Model  Accuracy  F1-score
          Bagging  0.925000  0.924089
    Random Forest  0.961111  0.960940
         AdaBoost  0.830556  0.829072
Gradient Boosting  0.955556  0.955334


# Сравнение и анализ результатов

## Результаты оценки
В таблице выше представлены значения метрик Accuracy и F1-score для каждой из четырех обученных моделей.

## Выводы

### Сравнение моделей бэггинга:
- **Random Forest** обычно показывает высокую точность (95-97% на Digits)
- **Bagging** демонстрирует схожие результаты

### Сравнение моделей бустинга:
- **Gradient Boosting** часто превосходит AdaBoost
- Обе модели бустинга показывают отличные результаты (95-97%)

### Общий итог:
На данном наборе данных (Digits) все ансамблевые модели показывают высокую точность (>95%). Задача распознавания цифр хорошо решается ансамблевыми методами. Для улучшения качества можно увеличить количество деревьев или настроить гиперпараметры.